# Project: Customer Churn Prediction

> Part of the **Machine Learning Learning Series**

## Project Overview
**Goal**: Predict which customers will churn (cancel service).

**Skills demonstrated**: Binary classification, imbalanced data, feature importance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.utils import resample
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'tenure': np.random.randint(1, 72, n),
    'monthly_charges': np.random.uniform(20, 120, n),
    'total_charges': np.random.uniform(100, 8000, n),
    'contract': np.random.choice(['Month-to-month','One year','Two year'], n, p=[0.55,0.25,0.20]),
    'internet_service': np.random.choice(['DSL','Fiber optic','No'], n),
    'churn': np.random.choice([0,1], n, p=[0.73,0.27])
})
print(df.shape)
print('Churn rate:', df['churn'].mean().round(3))

## 1. EDA — Class Imbalance

In [ ]:
df['churn'].value_counts().plot(kind='bar', color=['steelblue','coral'])
plt.title('Churn Distribution')
plt.xlabel('Churn (0=No, 1=Yes)')
plt.show()

## 2. Preprocessing

In [ ]:
le = LabelEncoder()
df_encoded = df.copy()
for col in ['contract','internet_service']:
    df_encoded[col] = le.fit_transform(df[col])

X = df_encoded.drop('churn', axis=1)
y = df_encoded['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

## 3. Model Training

In [ ]:
for name, model in [('Logistic Regression', LogisticRegression(class_weight='balanced', max_iter=1000)),
                     ('Random Forest', RandomForestClassifier(100, class_weight='balanced', random_state=42)),
                     ('Gradient Boosting', GradientBoostingClassifier(100, random_state=42))]:
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:,1]
    auc = roc_auc_score(y_test, y_prob)
    print(f'\n{name} — AUC: {auc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['No Churn','Churn']))